# 🌲 Notebook 01 — Průzkum a příprava dat
## Kombinovaný styl: Ukázka vzoru → Tvůj úkol

---
Každá sekce je rozdělena na dvě části:
- 🔵 **[UKÁZKA]** — kompletní kód, který si prostuduj a spusť
- 🟡 **[TEĎ TY]** — analogický úkol s jinými daty/sloupci — napiš kód sama

Cílem je aby ses naučila vzor a pak ho aplikovala.

---
## ⚙️ Setup

Tuto buňku spusť vždy jako první — nastaví prostředí.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Setup hotový!')

---
## Sekce 1: Načtení a průzkum EKC datasetu (= Úloha 1)

### 🔵 [UKÁZKA] Načtení `forest_ekc_model.csv`


In [ ]:
# Vzor: Takhle se načte CSV a prozkoumají základní vlastnosti
forest_ekc = pd.read_csv('../../data_dan/forest_ekc_model.csv')

print(f'=== forest_ekc_model.csv ===')
print(f'Počet řádků:  {forest_ekc.shape[0]}')
print(f'Počet sloupců: {forest_ekc.shape[1]}')
print(f'Sloupce: {forest_ekc.columns.tolist()}')
print(f'Chybějící hodnoty:\n{forest_ekc.isnull().sum()}')
print()
forest_ekc.describe()

### 🟡 [TEĎ TY] Načti `Forest_share_1990_2025.csv`

**Zadání**: Stejným způsobem načti soubor `../../data_raw/Forest_share_1990_2025.csv` a prozkoumej ho:
- Uloži do proměnné `forest_share`
- Vypiš počet řádků, sloupce, chybějící hodnoty a `.describe()`

<details>
<summary>💡 Nápověda</summary>

Vzor máš výše — stačí změnit cestu k souboru a název proměnné.

</details>

> ✅ **Kontrolní bod**: Počet řádků by měl být **200+**. Sloupce by měly obsahovat kód země a numerické sloupce s hodnotami lesního pokryvu. Pokud vidíš `KeyError` nebo prázdný DataFrame, zkontroluj cestu k souboru.

In [ ]:
# TVŮJ KÓD ZDE

# forest_share = pd.read_csv('../../data_raw/Forest_share_1990_2025.csv')
# print(f'Počet řádků: {forest_share.shape[0]}')
# print(f'Sloupce: {forest_share.columns.tolist()}')
# print(f'Chybějící hodnoty:\n{forest_share.isnull().sum()}')
# print()
# forest_share.describe()

---
## Sekce 2: Filtrování a podmíněný výběr (= Úloha 2)

### 🔵 [UKÁZKA] Kolik zemí zalesnilo vs. odlesnilo?


In [ ]:
# Vzor: Podmíněné filtrování a počítání
# (df['sloupec'] > 0) vrátí True/False pro každý řádek
# .sum() spočítá kolik True je

zalesnovani = (forest_ekc['Forest_change'] > 0).sum()
odlesnovani = (forest_ekc['Forest_change'] < 0).sum()
bez_zmeny   = (forest_ekc['Forest_change'] == 0).sum()

print(f'Zalesňující (Forest_change > 0): {zalesnovani}')
print(f'Odlesňující (Forest_change < 0): {odlesnovani}')
print(f'Beze změny  (Forest_change = 0): {bez_zmeny}')

# Zobraz 3 země s největší změnou
print('\nTop 3 nejvíce zalesnění:')
print(forest_ekc.nlargest(3, 'Forest_change')[['Country', 'Forest_change']].to_string(index=False))
print('\nTop 3 nejvíce odlesnění:')
print(forest_ekc.nsmallest(3, 'Forest_change')[['Country', 'Forest_change']].to_string(index=False))

### 🟡 [TEĎ TY] Analýza na `forest_share` datasetu

**Zadání**: Pro `forest_share` dataset zjisti:
1. Kolik zemí má v roce 2025 **více lesa** než v roce 1990 (porovnej sloupce)
2. Vypiš 3 země s největším lesním pokryvem v roce 2025
3. Vypiš 3 země s nejmenším lesním pokryvem v roce 2025

<details>
<summary>💡 Nápověda: názvy sloupců</summary>

Nejdřív vypiš `forest_share.columns.tolist()` aby sis zjistila přesné názvy sloupců.

</details>

> ✅ **Kontrolní bod**: Výsledky by měly dávat smysl geograficky — největší lesní pokryv mají tropické země (Surinam, Gabon, Guyana, Mikronésie...), nejmenší pouštní a polární oblasti. Pokud vidíš nerozumné hodnoty (záporná čísla, hodnoty přes 100), zkontroluj datové typy.

In [ ]:
# TVŮJ KÓD ZDE

# print(forest_share.columns.tolist())
# col_1990 = forest_share.columns[1]  # 'Forest share 1990 (%)'
# col_2025 = forest_share.columns[2]  # 'Forest share 2025 (%)'

# vic_lesa = (forest_share[col_2025] > forest_share[col_1990]).sum()
# print(f'Zemí s více lesa v 2025 než v 1990: {vic_lesa}')

# print('\nTop 3 největší lesní pokryv v 2025:')
# print(forest_share.nlargest(3, col_2025)[['Entity', col_2025]].to_string(index=False))
# print('\nTop 3 nejmenší lesní pokryv v 2025:')
# print(forest_share.nsmallest(3, col_2025)[['Entity', col_2025]].to_string(index=False))

---
## Sekce 3: Skupinová agregace — groupby (= Úlohy 3a–3d)

### 🔵 [UKÁZKA] Průměrné HDP podle skupin

Nejprve načteme panelová data a vypočítáme průměr.

> 📋 **Panelová data**: `MAIN_Forest_GDP_joined.csv` má strukturu **Country × Year** — každá země má jeden řádek pro každý rok (1990–2024), celkem ~7 000 řádků. Abychom dostali **jedno číslo HDP na zemi**, seskupíme (`groupby`) podle země a vezmeme průměr přes roky. Výsledek bude ~200 řádků — jeden na zemi.

> 📝 **Proč průměr a ne HDP za konkrétní rok?** Změna lesa probíhala **postupně** v průběhu 35 let — proto průměrné HDP lépe zachytí ekonomickou úroveň v celém sledovaném období než snapshot jediného roku. Alternativy (HDP roku 2000, medián) dávají podobné výsledky. ⚠️ Omezení: zemím s rychlým růstem (Vietnam, Čína) přiřadíme nižší HDP, než mají dnes.


In [ ]:
# Vzor: Načtení, filtrování a groupby agregace
forest_gdp = pd.read_csv('../../data_dan/MAIN_Forest_GDP_joined.csv')
forest_gdp = forest_gdp.rename(columns={
    'Share of land covered by forest': 'forest_pct',
    'GDP per capita (current US$)': 'gdp_per_capita'
})
forest_gdp_filtered = forest_gdp[forest_gdp['Year'] >= 1990].copy()

# .groupby(['Entity', 'Code']) seskupí záznamy se stejnou zemí
# ['gdp_per_capita'].mean() spočítá průměr pro každou skupinu
# .reset_index() převede zpět na normální DataFrame
mean_gdp = (
    forest_gdp_filtered
    .groupby(['Entity', 'Code'])['gdp_per_capita']
    .mean()
    .reset_index()
    .rename(columns={'gdp_per_capita': 'mean_gdp', 'Entity': 'Country'})
)

print(f'Průměrné HDP pro {len(mean_gdp)} zemí')
print('\nTop 5 zemí s nejvyšším průměrným HDP:')
print(mean_gdp.nlargest(5, 'mean_gdp')[['Country', 'mean_gdp']].to_string(index=False))

### 🟡 [TEĎ TY] Průměrný lesní pokryv podle let

**Zadání**: Z datasetu `forest_gdp_filtered` spočítej **průměrný i medián % lesního pokryvu (`forest_pct`) pro každý rok** (1990–2024):
1. Použij `.groupby('Year')['forest_pct'].agg(mean='mean', median='median', count='count').reset_index().round(3)`
2. Vypiš hodnoty sloupce `mean` pro roky 1990, 2000, 2010, 2020, 2024
3. Graf je připraven v buňce níže — spusť ho a ověř, že trend odpovídá hodnotám z bodu 2

<details>
<summary>💡 Nápověda: groupby</summary>

Seskupuješ jen podle 1 sloupce (Year), ne 2 (Entity + Code). Výsledný DataFrame bude mít sloupce: `Year`, `mean`, `median`, `count`.

</details>

**Očekávaný výstup (ukázka hodnot):**
```
Roky v datech: 1990 – 2025

Průměrný % lesa (vybrané roky):
  1990: xx.xx%
  2000: xx.xx%
  2010: xx.xx%
  2020: xx.xx%
  2024: xx.xx%
```

> ⚠️ **Metodická poznámka**: Toto je **nevážený průměr** — každá země (Monaco i Rusko) má stejnou váhu bez ohledu na rozlohu. Nezobrazuje skutečné globální procento plochy lesa, ale typickou hodnotu pro průměrnou zemi.

> 🔍 Pokud průměrný % lesa postupně klesá — svět jako celek odlesňuje. Jestli stoupá nebo je stabilní, pátrej po tom proč (změna měření, jiný vzorek zemí?).

> 📌 Skutečné globální procento (vážený průměr rozlohou zemí) spočítáme v dalším [UKÁZKA] bloku níže (stále Sekce 3).

In [ ]:
# TVŮJ KÓD ZDE

# yearly_forest = (
#     forest_gdp_filtered
#     .groupby('Year')['forest_pct']
#     .agg(mean='mean', median='median', count='count')
#     .reset_index()
#     .round(3)
# )

# print(f'Roky v datech: {yearly_forest["Year"].min()} – {yearly_forest["Year"].max()}')
# print(f'\nPrůměrný % lesa (vybrané roky):')
# for yr in [1990, 2000, 2010, 2020, 2024]:
#     row = yearly_forest[yearly_forest['Year'] == yr]
#     if len(row):
#         print(f'  {yr}: {row.iloc[0]["mean"]:.2f}%')

In [ ]:
# Grafická kontrola — čárový graf (kód připraven předem)
# Spusť tuto buňku až po dokončení kroků 1–2 výše (musí existovat: yearly_forest)

if 'yearly_forest' not in dir():
    print('⚠️  Nejdřív dokonči předchozí buňku — yearly_forest není definováno!')
elif 'mean' not in yearly_forest.columns and 'forest_pct' not in yearly_forest.columns:
    print('⚠️  yearly_forest musí mít sloupec "mean" (z .agg()) nebo "forest_pct" (z .mean()).')
    print('    Použij: .agg(mean="mean", median="median", count="count").reset_index()')
else:
    y_col = 'mean' if 'mean' in yearly_forest.columns else 'forest_pct'
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(yearly_forest['Year'], yearly_forest[y_col], 'b-', linewidth=2, label='Průměr')
    if 'median' in yearly_forest.columns:
        ax.plot(yearly_forest['Year'], yearly_forest['median'], 'g--', linewidth=1.5, label='Medián')
        ax.fill_between(yearly_forest['Year'], yearly_forest[y_col], yearly_forest['median'], alpha=0.1)
        ax.legend()
    ax.set_xlabel('Rok')
    ax.set_ylabel('% lesního pokryvu')
    ax.set_title('Světový průměr % lesního pokryvu 1990–2024')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('\n➡️  Pokud průměr klesá, globálně přichází o les. Pokud roste, globálně zalesňujeme.')

---
### 🔵 [UKÁZKA] Skutečné globální % lesa — vážený průměr

Nevážený průměr výše dává každé zemi stejnou váhu. Soubor `Forest area compared to total area.csv` obsahuje **absolutní plochy** (v 1 000 ha), ze kterých spočítáme skutečné globální procento.

> ⚠️ **Dvouřádkové záhlaví**: Tento soubor má nestandardní strukturu — řádek 0 je typ dat, řádek 1 je rok. Proto musíme načíst s `header=[0, 1]` místo obvyklého `header=0`.

In [ ]:
# Vzor: Načtení CSV se dvouřádkovým záhlavím + vážený průměr pro rok 1990
fa = pd.read_csv('../../data_raw/Forest area compared to total area.csv',
                 header=[0, 1], index_col=0)

forest_sum = fa['Forest (1 000 ha)']['1990'].dropna().sum()
land_sum   = fa['Total land area (1 000 ha)']['1990'].dropna().sum()
weighted_1990    = forest_sum / land_sum * 100
unweighted_1990  = forest_gdp_filtered[forest_gdp_filtered['Year'] == 1990]['forest_pct'].mean()

print('Rok 1990:')
print(f'  Vážený průměr  (skutečné globální %): {weighted_1990:.2f}%')
print(f'  Nevážený průměr (typická země):       {unweighted_1990:.2f}%')
print(f'  Rozdíl: {weighted_1990 - unweighted_1990:+.2f} pp')
# Output:
# Rok 1990:
#   Vážený průměr  (skutečné globální %): 33.33%
#   Nevážený průměr (typická země):       34.25%
#   Rozdíl: -0.92 pp

### 🟡 [TEĎ TY] Vážený průměr pro všechny dostupné roky

**Zadání**: Rozšiř ukázku na všechny dostupné roky (1990, 2000, 2010, 2015, 2020, 2025):
1. Pro každý rok spočítej `weighted_pct = forest_sum / land_sum * 100`
2. Vypiš tabulku se všemi roky
3. Porovnej hodnoty 1990 a 2025 s nevážným průměrem z `forest_ekc` (sloupce `Forest_1990`, `Forest_2025`)

<details>
<summary>💡 Nápověda: jak číst sloupce</summary>

Po `fa = pd.read_csv(..., header=[0,1], index_col=0)` přistupuješ ke sloupci takto:
`fa['Forest (1 000 ha)']['1990']` → série hodnot pro rok 1990.

Dostupné roky (jako řetězce): `'1990'`, `'2000'`, `'2010'`, `'2015'`, `'2020'`, `'2025'`

</details>

**Očekávaný výstup:**
```
Skutečný vážený průměr lesního pokryvu:
1990: 33.33%
2000: 32.51%
2010: 32.24%
2015: 32.09%
2020: 31.96%
2025: 31.79%

Srovnání s nevážným průměrem per-country:
  1990: nevážený 34.18% vs. vážený 33.33% → -0.86 pp
  2025: nevážený 32.97% vs. vážený 31.79% → -1.18 pp
```

> ℹ️ **Proč 34.18% a ne 34.25% jako v UKÁZCE výše?** UKÁZKA vypočítala nevážený průměr z panelových dat pro rok 1990 (`forest_gdp_filtered`). Zde porovnáváme s průměrem ze snapshotového datasetu `forest_ekc` (sloupce `Forest_1990`/`Forest_2025`), který pokrývá všechna léta 1990–2025. Obě čísla jsou "nevážené průměry přes země", ale z mírně odlišných zdrojů — odtud rozdíl 0.07 pp.

> 🔍 **Závěr**: Vážený průměr je o ~1 pp nižší než nevážený — velké, hustě zalesněné země (Rusko, Kanada, Brazílie) mají ve váženém průměru větší vliv. Obě řady ale ukazují shodný klesající trend.

In [ ]:
# TVŮJ KÓD ZDE

# fa = pd.read_csv('../../data_raw/Forest area compared to total area.csv',
#                  header=[0, 1], index_col=0)

# roky = ['1990', '2000', '2010', '2015', '2020', '2025']
# print('Skutečný vážený průměr lesního pokryvu:')
# weighted = {}
# for rok in roky:
#     forest_sum = fa['Forest (1 000 ha)'][rok].dropna().sum()
#     land_sum   = fa['Total land area (1 000 ha)'][rok].dropna().sum()
#     weighted[rok] = forest_sum / land_sum * 100
#     print(f'{rok}: {weighted[rok]:.2f}%')

## Srovnání s nevážným průměrem:
# unweighted_1990 = forest_ekc['Forest_1990'].mean()
# unweighted_2025 = forest_ekc['Forest_2025'].mean()
# print(f'\nSrovnání s nevážným průměrem per-country:')
# print(f'  1990: nevážený {unweighted_1990:.2f}% vs. vážený {weighted["1990"]:.2f}% → {weighted["1990"] - unweighted_1990:+.2f} pp')
# print(f'  2025: nevážený {unweighted_2025:.2f}% vs. vážený {weighted["2025"]:.2f}% → {weighted["2025"] - unweighted_2025:+.2f} pp')

---
## Sekce 4: Klasifikace Světové banky (= Úloha 4)

### 🔵 [UKÁZKA] Načtení CSV se záhlavím na jiném řádku — `skiprows`

Soubory s metadatovými řádky před záhlavím vyžadují parametr `skiprows`. Bez něj pandas přečte metadata jako záhlaví a dostaneme `KeyError: 'Code'`.

In [ ]:
# Vzor: skiprows=2 přeskočí 2 metadatové řádky, teprve třetí řádek je záhlaví
# Soubor 2025_World_Bank_classification_by_Income.csv:
#   Řádek 0: "Updated: 2025"        ← metadata, přeskoč
#   Řádek 1: "Classification ..."   ← metadata, přeskoč
#   Řádek 2: "Country","Code",...   ← záhlaví sloupců  ← čti od tady (skiprows=2)
#   Řádek 3+: data

wb_demo = pd.read_csv('../../data_raw/2025_World_Bank_classification_by_Income.csv', skiprows=2, nrows=3)
print("S skiprows=2 — záhlaví je správné:")
print(wb_demo[['Code', 'Country', 'Region', 'Income group']].to_string(index=False))

### 🟡 [TEĎ TY] Načtení klasifikace Světové banky (= Úloha 4)

**Zadání**:
1. Načti `../../data_raw/2025_World_Bank_classification_by_Income.csv` do `wb_raw` pomocí `skiprows=2`
2. Vyber sloupce `Code`, `Region`, `Income group` → ulož jako `wb_class`
3. Přejmenuj na malá písmena: `code`, `region`, `income_group`
4. Odstraň řádky kde `code` nebo `income_group` je NaN
5. Filtruj pouze validní ISO3 kódy: `wb_class = wb_class[wb_class['code'].str.match(r'^[A-Z]{3}$', na=False)]`
6. Vypiš počet zemí a `.value_counts()` pro `income_group`

<details>
<summary>💡 Proč str.match(r'^[A-Z]{3}$')?</summary>

Dataset obsahuje kromě zemí i řádky pro regiony a agregáty. Regex `^[A-Z]{3}$` znamená: začátek (`^`), přesně 3 velká písmena, konec (`$`). `na=False` ignoruje NaN bez chyby.

</details>

**Očekávaný výstup:**
```
Počet zemí: 218
income_group
High income            86
Upper middle income    55
Lower middle income    51
Low income             26
```

In [ ]:
# TVŮJ KÓD ZDE — načtení WB klasifikace (kroky 1–6)

# wb_raw = pd.read_csv('../../data_raw/2025_World_Bank_classification_by_Income.csv', skiprows=2)
# wb_class = wb_raw[['Code', 'Region', 'Income group']].copy()
# wb_class.columns = ['code', 'region', 'income_group']
# wb_class = wb_class.dropna(subset=['code', 'income_group'])
# wb_class = wb_class[wb_class['code'].str.match(r'^[A-Z]{3}$', na=False)]
# print(f'Počet zemí: {len(wb_class)}')
# print(wb_class['income_group'].value_counts())

---
## Sekce 5: Spojení tabulek — merge (= Úlohy 5a+5b)

### 🔵 [UKÁZKA] Propojení forest_ekc s průměrným HDP


In [ ]:
# Vzor: pd.merge() propojí dvě tabulky přes společný klíč
# how='left' → zachovej všechny řádky z levé tabulky
# on='Code' → propoj přes sloupec 'Code'

ekc_with_gdp = pd.merge(
    forest_ekc,
    mean_gdp[['Code', 'mean_gdp']],
    on='Code',
    how='left'
)

print(f'Po merge: {len(ekc_with_gdp)} zemí, {ekc_with_gdp.shape[1]} sloupců')
print(f'Chybějící HDP: {ekc_with_gdp["mean_gdp"].isnull().sum()} zemí')
ekc_with_gdp.head(3)

### 🟡 [TEĎ TY] Kompletní master dataset

> 📌 `wb_class` jsi načetla v Sekci 4. Teď ho propoj s `ekc_with_gdp` z UKÁZKY výše.

**Zadání** — dvě buňky níže, každá pro jednu část:

**Buňka 1 (kroky 1–2): Merge s wb_class:**
1. Sluč `ekc_with_gdp` s `wb_class` — klíče mají **různou velikost písmen**:
   → Použij `left_on='Code', right_on='code'`
2. **Hned po merge** odstraň přebytečný klíč: `ekc_master = ekc_master.drop(columns=['code'])`

<details>
<summary>💡 Proč drop hned tady?</summary>

`pd.merge(..., left_on='Code', right_on='code')` zachová oba klíče. Pokud `code` neodstraníš teď a v kroku 3 přejmenuješ `Code` → `code`, budeš mít dva sloupce `code` — a pandas pak vyhodí záhadnou chybu o deset kroků dál.

</details>

> ✅ **Kontrolní bod**:
> ```python
> print(f'Po merge: {ekc_master.shape}')  # ~ (211, 8)
> ```

**Buňka 2 (kroky 3–5): Vyčistění datasetu:**
3. Přejmenuj sloupce na malá písmena (Country→country, Code→code, Forest_1990→forest_1990, atd.)
4. Odstraň řádky kde chybí `forest_change`, `mean_gdp` nebo `income_group` → ulož jako `ekc_complete`
5. Přidej sloupec `log_gdp = np.log(mean_gdp)`

> ✅ **Kontrolní bod (buňka 2)**: Po spuštění by měl `ekc_complete` mít přibližně **199 zemí**. Zkontroluj: `print(f'Kompletní záznamy: {len(ekc_complete)}')` — pokud máš výrazně jiné číslo, zkontroluj merge nebo dropna v kroku 4.

In [ ]:
# TVŮJ KÓD ZDE — kroky 1–2: merge s wb_class

# ekc_master = pd.merge(
#     ekc_with_gdp,
#     wb_class,
#     left_on='Code',
#     right_on='code',
#     how='left'
# )
# ekc_master = ekc_master.drop(columns=['code'])
# print(f'Po merge: {ekc_master.shape}')  # ~ (211, 8)

In [ ]:
# TVŮJ KÓD ZDE — kroky 3–5: přejmenování, dropna, log_gdp

# ekc_master = ekc_master.rename(columns={
#     'Country': 'country', 'Code': 'code',
#     'Forest_1990': 'forest_1990', 'Forest_2025': 'forest_2025', 'Forest_change': 'forest_change'
# })
# ekc_complete = ekc_master.dropna(subset=['forest_change', 'mean_gdp', 'income_group']).copy()
# ekc_complete['log_gdp'] = np.log(ekc_complete['mean_gdp'])
# print(f'Kompletní záznamy: {len(ekc_complete)} zemí')

---
## Sekce 6: Skupinová statistika s vizualizací (= Úloha 6)

### 🔵 [UKÁZKA] Statistiky podle regiónu


In [ ]:
# Nejprve si načteme ekc_complete (nebo použijeme ekc_analysis.csv)
try:
    ekc_complete  # Pokud existuje z kroku výše
except NameError:
    ekc_complete = pd.read_csv('../output/ekc_analysis.csv')
    if 'log_gdp' not in ekc_complete.columns:
        ekc_complete['log_gdp'] = np.log(ekc_complete['mean_gdp'])

# Vzor: Skupinová statistika s více sloupci
region_stats = (
    ekc_complete
    .groupby('region')['forest_change']     # Seskup podle regionu
    .agg(['mean', 'median', 'std', 'count']) # Spočítej statistiky
    .sort_values('mean')                     # Seřaď vzestupně
    .round(3)
)
region_stats.columns = ['Průměr', 'Medián', 'Std', 'Počet zemí']
print(region_stats)

### 🟡 [TEĎ TY] Statistiky podle příjmové skupiny

**Zadání**:
1. Stejným způsobem spočítej statistiky pro `income_group` (ne region)
2. Seřaď výsledek pomocí `.reindex(['Low income', 'Lower middle income', 'Upper middle income', 'High income'])`
3. Přejmenuj sloupce na češtinu: `.columns = ['Průměr', 'Medián', 'Std', 'Počet zemí']`
4. Graf je připraven v buňce níže (za Sanity Checkem) — spusť ho a ověř, že hodnoty souhlasí

<details>
<summary>💡 Nápověda: seřazení skupin</summary>

`.reindex([...])` přeuspořádá řádky do zvoleného pořadí — stejný výsledek jako `.sort_values()` ale bez třídění podle hodnot.

</details>

**Očekávaný výstup (formát):**
```
                     Průměr  Medián   Std  Počet zemí
income_group
Low income           -5.547  -2.880  7.005          25
Lower middle income  -3.911  -1.140  8.631          47
Upper middle income  -0.608   0.000  7.406          52
High income           1.383   0.790  4.287          75
```

> 🔍 **Klíčový výsledek**: Jediná skupina s kladným průměrem je **High income** (+1.38 %). Trend je monotónní — čím bohatší skupina, tím méně odlesňuje. Toto je vizuální důkaz EKC trendu v datech.

In [ ]:
# TVŮJ KÓD ZDE — statistiky podle income_group

# income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
# income_summary = (
#     ekc_complete
#     .groupby('income_group')['forest_change']
#     .agg(['mean', 'median', 'std', 'count'])
#     .reindex(income_order)
#     .round(3)
# )
# income_summary.columns = ['Průměr', 'Medián', 'Std', 'Počet zemí']
# income_summary

---
## Bonus: Sanity Check — Ověření správnosti dat

Před analýzou vždy zkontrolujeme, že merge dopadl správně.
Vybereme několik konkrétních zemí a ověříme, že hodnoty dávají smysl.

In [ ]:
# Ověření: Jsou klíčové země v datasetu se správnými hodnotami?
test_codes = ['CZE', 'BRA', 'ETH', 'DEU', 'IDN']
print('=== SANITY CHECK ===')
for code in test_codes:
    row = ekc_complete[ekc_complete['code'] == code]
    if len(row) == 0:
        print(f'CHYBA: {code} nenalezen! Zkontroluj merge.')
    else:
        r = row.iloc[0]
        print(f'{code} ({r["country"]:20s}) | income: {str(r.get("income_group","?")):<22} | '
              f'forest_change: {r["forest_change"]:+6.2f}% | HDP: ${r["mean_gdp"]:>9,.0f}')

# Očekávané hodnoty (přibližně):
# CZE (Czech Republic) → High income, forest mírně kladný
# BRA (Brazil)         → Upper middle income, velká plocha lesa
# ETH (Ethiopia)       → Low income, malé HDP
# DEU (Germany)        → High income, stabilní les
# IDN (Indonesia)      → Lower middle income, odlesnění

n_high = (ekc_complete['income_group'] == 'High income').sum()
n_low  = (ekc_complete['income_group'] == 'Low income').sum()
print(f'\nHigh income zemí: {n_high} | Low income zemí: {n_low}')
if n_high == 0 or n_low == 0:
    print('POZOR: Některá příjmová skupina má 0 zemí — problém s merging!')
else:
    print('Rozložení příjmových skupin vypadá v pořádku.')

In [ ]:
# Grafická kontrola — barplot (kód připraven předem)
# Spusť tuto buňku až po dokončení statistik výše (musí existovat: ekc_complete)

if 'ekc_complete' not in dir():
    print('⚠️  Nejdřív dokonči předchozí buňky — ekc_complete není definováno!')
else:
    income_order_plot = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
    means_plot = (
        ekc_complete.groupby('income_group')['forest_change']
        .mean()
        .reindex(income_order_plot)
    )

    colors_bar = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c']
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(range(len(means_plot)), means_plot.values,
                  color=colors_bar, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xticks(range(len(means_plot)))
    ax.set_xticklabels(means_plot.index, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Průměrná změna lesního pokryvu [pp]')
    ax.set_title('Průměrná změna lesa podle příjmové skupiny (1990–2025)')
    for bar, val in zip(bars, means_plot.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (0.1 if val >= 0 else -0.2),
                f'{val:+.2f}', ha='center',
                va='bottom' if val >= 0 else 'top', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## Sekce 7: Export (= Úloha 7)

### 🟡 [TEĎ TY] Export výsledného datasetu

**Zadání**:
1. Vytvoř složku `../output/` (použij `os.makedirs('../output', exist_ok=True)`)
2. Exportuj `ekc_complete` do `../output/ekc_analysis.csv` bez index sloupce
   - Použij `encoding='utf-8-sig'` (pro správné zobrazení češtiny v Excelu)

**Očekávaný výstup:**
```
✅ Uloženo: ../output/ekc_analysis.csv (199 řádků)
```

> ✅ **Ověření**: Otevři soubor v Excelu — sloupce by měly být: `country`, `code`, `forest_1990`, `forest_2025`, `forest_change`, `mean_gdp`, `region`, `income_group`, `log_gdp`.

In [ ]:
# TVŮJ KÓD ZDE

# os.makedirs('../output', exist_ok=True)
# ekc_complete.to_csv('../output/ekc_analysis.csv', index=False, encoding='utf-8-sig')
# print(f'✅ Uloženo: ../output/ekc_analysis.csv ({len(ekc_complete)} řádků)')

---
## ✅ Hotovo?

Zkontroluj:
- [ ] Soubor `../output/ekc_analysis.csv` existuje
- [ ] Má sloupce: `country`, `code`, `forest_change`, `mean_gdp`, `log_gdp`, `income_group`, `region`

**Pokračuj**: `02_statistika_ekc.ipynb`
